# Video Generation with Sound using LTX-2.3 and ComfyUI on AMD Radeon PRO W7900 GPUs

With the rapid development of AI-generated content (AIGC) technologies, text-to-video generation has become a powerful tool for creators, researchers, and media professionals.

[ComfyUI](https://github.com/comfyanonymous/ComfyUI) is a node-based interface for building image and video generation workflows. Instead of writing generation code from scratch, users can connect modular nodes to load models, configure prompts, run inference, and export results.

[LTX-2.3](https://huggingface.co/Lightricks/LTX-2.3) is Lightricks’ open-weight audio-video generation model. It is designed to generate synchronized video and audio in a single model, making it suitable for hands-on text-to-video-with-sound workflows.

In this workshop, you will run **LTX-2.3** with **ComfyUI** on **AMD Radeon PRO W7900** GPUs using ROCm software. You will learn how to launch ComfyUI inside this notebook environment, load the required workflow, enter a text prompt, and generate a short video with sound on AMD hardware.

## Prerequisites

This workshop was developed and tested with the following environment.

### Hardware

- **AMD Radeon PRO W7900 GPU**
  - GPU architecture: **AMD RDNA™ 3**
  - Dedicated memory: **48 GB VRAM**
  - Peak FP16 performance: 123 TFLOPs

### Software

The container image used for this workshop includes the following components:

- **ROCm 7.2.4**
- **PyTorch 2.10 with ROCm/HIP support**
- **ComfyUI 0.25.0**
- **LTX-2.3 BF16 model files**

### Verify AMD GPU Availability

Before starting ComfyUI, confirm that the AMD GPU is visible inside the notebook environment.
```

In [ ]:
!amd-smi

This command lists available AMD GPUs along with their key details (VRAM, power, temperature).


# ComfyUI Setup

In this section, you will verify the PyTorch ROCm environment and prepare to launch ComfyUI for LTX-2.3 video generation with sound.

## Verify the PyTorch ROCm Installation

Before starting ComfyUI, verify that PyTorch is correctly installed and can access the AMD GPU through ROCm/HIP.

> **Note:** In ROCm-enabled PyTorch builds, GPU APIs are still exposed through the `torch.cuda` namespace for compatibility.  
> Seeing `torch.cuda` in the checks below is expected, even when running on AMD GPUs.

Run the following cell to check:

- the installed PyTorch version
- the ROCm/HIP runtime version
- whether a GPU is available


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"ROCm/HIP version: {torch.version.hip}")
print(f"GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print("The GPU device is available.")
else:
    print("No GPU device was detected by PyTorch.")

The expected output should be similar to:

```text
PyTorch version: 2.10.x
ROCm/HIP version: 7.2.x
The GPU device is avaliable.
```


## Verify the ComfyUI Installation

ComfyUI is pre-installed in this workshop container.

- Installation path: `/comfyui_workspace/ComfyUI`
- Main entry point: `/comfyui_workspace/ComfyUI/main.py`
- Version file: `/comfyui_workspace/ComfyUI/comfyui_version.py`

Run the following cell to confirm that ComfyUI is available and ready to launch.


In [ ]:
from pathlib import Path

COMFYUI_DIR = Path("/comfyui_workspace/ComfyUI")
MAIN_FILE = COMFYUI_DIR / "main.py"
VERSION_FILE = COMFYUI_DIR / "comfyui_version.py"

if MAIN_FILE.exists():
    print(f"ComfyUI found: {COMFYUI_DIR}")
else:
    raise FileNotFoundError(f"ComfyUI not found at {COMFYUI_DIR}")

if VERSION_FILE.exists():
    print("\nComfyUI version information:")
    print(VERSION_FILE.read_text())
else:
    print("\nComfyUI version file not found.")

# Running Video Generation with Sound

Follow the steps in this section to generate a short video with synchronized sound from a text prompt using **LTX-2.3** and **ComfyUI**.

---

## Model Setup

### 📦 Required Model Files

The provided ComfyUI workflow for **LTX-2.3** requires the following **5 BF16 model files**.

These files are pre-mounted into the workshop container, so no manual download is required.

| Role | File | Directory | Approx. Size |
|---|---|---|---|
| Main diffusion model | `ltx-2.3-22b-dev.safetensors` | `models/checkpoints/` | 46 GB |
| Text encoder | `gemma_3_12B_it.safetensors` | `models/text_encoders/` | 24 GB |
| Distilled LoRA | `ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors` | `models/loras/` | 2.7 GB |
| Gemma LoRA | `gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors` | `models/loras/` | 0.6 GB |
| Spatial upscaler | `ltx-2.3-spatial-upscaler-x2-1.1.safetensors` | `models/latent_upscale_models/` | 1 GB |

### 📂 Directory Structure

The model files should be available under the ComfyUI `models` directory:

```text
/comfyui_workspace/ComfyUI/
└──📂 models/
    ├──📂 checkpoints/
    │   └── ltx-2.3-22b-dev.safetensors
    ├──📂 text_encoders/
    │   └── gemma_3_12B_it.safetensors
    ├── 📂loras/
    │   ├── ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors
    │   └── gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors
    └──📂 latent_upscale_models/
        └── ltx-2.3-spatial-upscaler-x2-1.1.safetensors

## 🚀 Launch the ComfyUI Server

Now start the ComfyUI server and expose it through a temporary **Radeon Tunnel**.

Run the following cell and wait until you see a line similar to:

```text
GREEN Public URL: https://<unique-tunnel-url>
```

This **GREEN Public URL** is the link you will use to open **ComfyUI** in your browser.

In [ ]:
!bash ./start_comfyui_with_tunnel.sh

After the public URL appears:

1. Wait about **10 seconds** for the server and tunnel to finish initializing.
2. Open the printed **GREEN Public URL** in your browser.
3. Keep this notebook cell running while you use ComfyUI.

> **Important:** This cell runs continuously while the ComfyUI server is active.  
> Do **not** interrupt or stop the cell during the workshop.

### 🌐 Open the ComfyUI Interface

After the ComfyUI server starts, open the printed **GREEN Public URL** in your browser.

Use the printed **GREEN Public URL** only. 

### 🔄 Load the Workflow

In the ComfyUI interface:

1. Click **Workflows** in the left sidebar.
2. Select **LTX-2.3_t2v_BF16**.
3. Wait for the workflow to load on the canvas.

The workflow is already configured to use the pre-downloaded **LTX-2.3 BF16** model files.

### 📝 Update the Prompt

For your first run, you can keep the default settings and only update the **Prompt**. After that, you can also adjust the parameters below to customize the video generation.

Key default settings:

| Setting        |           Default | Description                                                          |
| -------------- | ----------------: | -------------------------------------------------------------------- |
| Prompt Enhance |            `True` | Automatically enhances the prompt with additional details.           |
| Width          |            `1280` | Sets the video width in pixels.                                      |
| Height         |             `720` | Sets the video height in pixels.                                     |
| Duration       |       `5 seconds` | Controls the length of the generated video.                          |
| FPS            |              `25` | Sets the number of frames generated per second.                      |
| Seed           | `810138461690240` | Controls randomness. Use the same seed to reproduce similar results. |

### ▶️ Run Generation

Click **Run**, or press **Ctrl + Enter** / **Cmd + Enter**.

Keep the ComfyUI browser tab open and do not stop the notebook cell running the server.

### 🎞️ Expected Output

The first generation may take about 3–4 minutes because the models need to be loaded. Once the models are loaded, generating a 5-second video usually takes about 1.5 minutes.

The output is a short video with sound, approximately:

- **1280 × 720**
- **25 fps**
- **5 seconds**

You can preview and download the video from the ComfyUI interface.

The output file is also saved under:

```text
/comfyui_workspace/ComfyUI/output/video/
```

## 🧠 Challenge: Create Your Own Audio-Video Story

Now it is your turn to create a short audio-video story using **LTX-2.3** in ComfyUI.

Your video should include:

- a clear visual action
- matching sound effects or ambience
- a creative story idea